# Lab 02: RAG Pipeline with ChromaDB + RAGAS Evaluation

**NCP-AAI Domain:** Knowledge Integration (12-15%)

## What You'll Build
A complete RAG (Retrieval-Augmented Generation) pipeline:
- Document ingestion with chunking strategies
- ChromaDB vector store with NVIDIA embeddings
- Retrieval with reranking
- RAGAS evaluation: Faithfulness, Context Precision, Context Recall, Answer Relevancy

## Learning Objectives
- Understand chunk size vs overlap trade-offs
- Implement cosine similarity retrieval
- Use a cross-encoder reranker
- Measure RAG quality with RAGAS metrics
- Diagnose high precision / low recall vs high recall / low precision

In [ ]:
!pip install -q chromadb langchain langchain-nvidia-ai-endpoints langchain-community ragas sentence-transformers

In [ ]:
import os
os.environ['NVIDIA_API_KEY'] = 'nvapi-xxx'  # Replace with your key

## Step 1: Prepare Documents and Chunking

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Sample corpus (in production: load from PDFs, web, databases)
corpus = [
    Document(
        page_content="""NVIDIA TensorRT-LLM is an open-source library for optimizing LLM inference.
        It supports FP8 quantization on H100 Hopper and Ada Lovelace GPUs only.
        For A100 and earlier GPUs, use INT8 SmoothQuant (W8A8) or INT4 AWQ (W4A16).
        In-flight batching (continuous batching) adds new requests and removes completed ones
        at every token generation step, maximizing GPU utilization.
        The paged KV cache splits KV cache into non-contiguous pages, reducing fragmentation
        and enabling prefix caching for common prompt prefixes.""",
        metadata={'source': 'tensorrt-llm-docs', 'topic': 'deployment'}
    ),
    Document(
        page_content="""LangGraph is a library for building stateful, cyclic multi-actor workflows.
        It uses three primitives: StateGraph (holds typed shared state), Nodes (Python functions
        that read/write state), and Edges (connections between nodes).
        The interrupt() function pauses graph execution and saves state to the checkpointer.
        Resume with graph.invoke(Command(resume=value), config) using the same thread_id.
        MemorySaver stores state in-memory. PostgresSaver stores in PostgreSQL for multi-replica deployments.""",
        metadata={'source': 'langgraph-docs', 'topic': 'architecture'}
    ),
    Document(
        page_content="""RAGAS evaluates RAG pipelines with four core metrics:
        Faithfulness: fraction of answer claims supported by retrieved context (detects hallucination).
        Context Precision: fraction of retrieved chunks that are actually relevant.
        Context Recall: fraction of needed information that was retrieved.
        Answer Relevancy: how directly the answer addresses the original question.
        High Precision + Low Recall = k too low or retrieval misses key chunks.
        Low Precision + High Recall = k too high, too many irrelevant chunks retrieved.""",
        metadata={'source': 'ragas-docs', 'topic': 'evaluation'}
    ),
    Document(
        page_content="""NeMo Guardrails uses Colang DSL (.co files) to define safety policies.
        Five rail types: Input rails (filter user messages before LLM), Output rails (filter LLM
        responses before user), Dialog rails (control conversation flow), Retrieval rails
        (filter retrieved documents), and Execution rails (wrap tool calls).
        Topical rails prevent off-topic conversations. Input rails block prompt injection.
        Colang syntax: 'define user ask <intent>' and 'define flow <name>' blocks.""",
        metadata={'source': 'nemo-guardrails-docs', 'topic': 'safety'}
    ),
    Document(
        page_content="""HNSW (Hierarchical Navigable Small World) achieves O(log N) approximate nearest neighbor
        search by navigating a multi-layer graph from sparse top layers to dense bottom layers.
        ChromaDB uses HNSW by default. Cosine similarity is the standard metric for text embeddings
        because it measures semantic direction independent of vector magnitude.
        After L2 normalization, cosine similarity equals dot product (faster to compute).
        HNSW parameters: M (connections per node), ef_construction (build quality), ef (search quality).""",
        metadata={'source': 'vector-db-docs', 'topic': 'knowledge'}
    ),
]

# Chunking with overlap
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,      # Characters per chunk
    chunk_overlap=50,    # Overlap to prevent context loss at boundaries
    separators=['\n\n', '\n', '. ', ' ', '']  # Split priority order
)

chunks = splitter.split_documents(corpus)
print(f'Original docs: {len(corpus)}')
print(f'After chunking: {len(chunks)} chunks')
print(f'\nSample chunk:\n{chunks[0].page_content}')

## Step 2: Build Vector Store with ChromaDB

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

# NVIDIA Embeddings via NIMs
embeddings = NVIDIAEmbeddings(
    model='NV-Embed-QA',  # NVIDIA's retrieval-optimized embedding model
    api_key=os.environ['NVIDIA_API_KEY'],
)

# Build ChromaDB vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='ncp-aai-knowledge-base',
    persist_directory='./chroma_db',  # Persist to disk
)

print(f'Vector store created with {vectorstore._collection.count()} vectors')

## Step 3: Retrieval with and without Reranking

In [ ]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers import ContextualCompressionRetriever

query = 'What quantization formats does TensorRT-LLM support on H100 GPUs?'

# --- Retrieval WITHOUT reranking ---
base_retriever = vectorstore.as_retriever(search_kwargs={'k': 5})
base_results = base_retriever.invoke(query)
print('=== Without Reranking (top 5) ===')
for i, doc in enumerate(base_results):
    print(f'{i+1}. [{doc.metadata["topic"]}] {doc.page_content[:100]}...')

print()

# --- Retrieval WITH cross-encoder reranker ---
reranker_model = HuggingFaceCrossEncoder(model_name='cross-encoder/ms-marco-MiniLM-L-6-v2')
compressor = CrossEncoderReranker(model=reranker_model, top_n=3)
reranking_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

reranked_results = reranking_retriever.invoke(query)
print('=== With Reranker (top 3 after reranking) ===')
for i, doc in enumerate(reranked_results):
    print(f'{i+1}. [{doc.metadata["topic"]}] {doc.page_content[:100]}...')

## Step 4: Full RAG Pipeline

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatNVIDIA(model='meta/llama-3.1-70b-instruct', temperature=0.0)

RAG_PROMPT = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the provided context. 
If the answer is not in the context, say 'I don't have information about that in my knowledge base.'
Do not use any prior knowledge.

Context:
{context}

Question: {question}

Answer:"""
)

def format_docs(docs):
    return '\n\n---\n\n'.join(doc.page_content for doc in docs)

rag_chain = (
    {'context': reranking_retriever | format_docs, 'question': RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# Test queries
queries = [
    'What quantization format does TensorRT-LLM support on H100 GPUs?',
    'How does interrupt() work in LangGraph?',
    'What does Faithfulness measure in RAGAS?',
]

for q in queries:
    print(f'Q: {q}')
    answer = rag_chain.invoke(q)
    print(f'A: {answer}')
    print()

## Step 5: RAGAS Evaluation

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

# Golden dataset: questions + ground truth answers
golden_dataset = [
    {
        'question': 'What quantization format does TensorRT-LLM support on H100 GPUs?',
        'ground_truth': 'TensorRT-LLM supports FP8 quantization on H100 Hopper and Ada Lovelace GPUs.',
    },
    {
        'question': 'What does RAGAS Faithfulness measure?',
        'ground_truth': 'Faithfulness measures the fraction of answer claims that are supported by the retrieved context, detecting hallucination.',
    },
    {
        'question': 'What is HNSW search time complexity?',
        'ground_truth': 'HNSW achieves O(log N) approximate nearest neighbor search time complexity.',
    },
]

# Generate answers and retrieve contexts for evaluation
eval_data = []
for item in golden_dataset:
    q = item['question']
    retrieved_docs = reranking_retriever.invoke(q)
    contexts = [doc.page_content for doc in retrieved_docs]
    answer = rag_chain.invoke(q)
    eval_data.append({
        'question': q,
        'answer': answer,
        'contexts': contexts,
        'ground_truth': item['ground_truth'],
    })

ds = Dataset.from_list(eval_data)

# Run RAGAS evaluation
results = evaluate(
    ds,
    metrics=[faithfulness, context_precision, context_recall, answer_relevancy]
)

print('=== RAGAS Evaluation Results ===')
print(results)
results_df = results.to_pandas()
print(results_df[['question', 'faithfulness', 'context_precision', 'context_recall', 'answer_relevancy']])

## Exam Recap

| Metric | High = Good | Low = Problem | Fix |
|--------|-------------|---------------|-----|
| Faithfulness | Answer grounded in context | Hallucination | Fix retrieval or prompt |
| Context Precision | Retrieved chunks are relevant | Noise in context | Reduce k or add reranker |
| Context Recall | All needed info retrieved | Missing information | Increase k or fix chunking |
| Answer Relevancy | Answer addresses the question | Off-topic answer | Fix prompt template |

## Challenge Exercises
1. Experiment with chunk_size=100 vs 500 — observe how Context Recall changes
2. Remove the reranker — measure impact on Context Precision
3. Add a 6th document about a completely different topic — observe how it affects retrieval noise
4. Implement hybrid search: combine BM25 keyword search + vector search results before reranking